# EDA 02: Prices, returns and regimes

**Price basis: raw closes, with each stock's returns dropped on every corporate-action ex-date.**
Adjusted prices (`data/derived/prices_adj`, task 2.4) don't exist yet. Every ex-date in `corporate_actions`
(all types, dividends included) is removed from that stock's daily returns, and any multi-day window that
contains one is dropped, so no return spans an unadjusted price change. When task 2.4 lands, re-run this
notebook on adjusted prices; the numbers should barely move.

- **Period:** 2016-01-01 to 2023-12-31. Train = 2016–2022, validation = 2023. Prices after 2023 are not
  loaded, so forward windows at the end of 2023 are empty rather than reaching into the test period.
- **Universe:** the current 50 Nifty stocks plus the Nifty 50 index (survivorship bias: stocks that left the
  index are missing). From E1: TATACONSUM starts 27 Feb 2020 and SHRIRAMFIN 20 Dec 2022 (old symbols not
  mapped yet); the trading sequence is the price dates, since the calendar misses 6 Muhurat sessions.
- **Returns:** log returns. Index returns come from closes, never the file's `change_pct` (wrong on 13 Mar 2023, E1).
- **Helpers:** `src/analysis/returns.py`, `src/features/regime.py` (settings: `regime` in config.yaml).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf, adfuller

from src.analysis import returns as R
from src.analysis.plots import heatmap, save
from src.config import Config
from src.features import regime as G

pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 4)
cfg = Config.load()
START, TRAIN_END, END = "2016-01-01", "2022-12-31", "2023-12-31"
HORIZONS = (1, 5, 10)
con = duckdb.connect(str(cfg.duckdb_path), read_only=True)

prices = con.execute("SELECT symbol, trade_date, open, high, low, close, volume, turnover FROM prices_daily "
                     "WHERE trade_date BETWEEN ? AND ?", [START, END]).df()
ca = con.execute("SELECT symbol, ex_date, ca_type FROM corporate_actions WHERE ex_date BETWEEN ? AND ?", [START, END]).df()
idx = con.execute("SELECT index_name, trade_date, open, high, low, close FROM index_daily WHERE trade_date BETWEEN ? AND ?",
                  [START, END]).df()
universe = pd.read_csv(cfg.manual_dir / "universe.csv")
sector = universe.set_index("symbol")["industry"]

O, H, L, C = (R.to_wide(prices, c) for c in ("open", "high", "low", "close"))
VOL, TURN = R.to_wide(prices, "volume"), R.to_wide(prices, "turnover")
MASK = R.ex_date_mask(C.index, C.columns, ca)
r = R.log_returns(C, MASK)

nifty = idx[idx.index_name == "Nifty 50"].set_index(pd.to_datetime(idx[idx.index_name == "Nifty 50"].trade_date)).sort_index()
vix = idx[idx.index_name == "India VIX"].set_index(pd.to_datetime(idx[idx.index_name == "India VIX"].trade_date)).sort_index().close
NC, NO = nifty[["close"]].rename(columns={"close": "NIFTY"}), nifty[["open"]].rename(columns={"open": "NIFTY"})
rn = R.log_returns(NC)["NIFTY"]
is_train = r.index <= TRAIN_END
TRAIN_STOCKS = r.columns[r[is_train].notna().sum() >= 250]     # stocks with a usable train history

print(f"{C.shape[1]} stocks, {len(C)} trading days ({C.index.min().date()} to {C.index.max().date()}); "
      f"stock-days with a price: {int(C.notna().sum().sum()):,}")
print(f"ex-dates dropped: {int(MASK.sum().sum())} ({MASK.sum().sum() / C.notna().sum().sum():.2%} of stock-days); "
      f"by type: {ca.ca_type.value_counts().to_dict()}")
print("Nifty days:", len(nifty), "| VIX days:", vix.notna().sum())

## 1. Return distributions at 1, 5 and 10 trading days

Close-to-close log returns; for 5 and 10 days only non-overlapping windows are used (every h-th day). `share_beyond_3sd` vs 0.27% for a normal distribution measures fat tails.

In [ ]:
rows, ret_h = [], {}
for h in HORIZONS:
    rs = R.label_returns(C, O, MASK, h, "close")
    ret_h[h] = rs
    rows.append({"series": "stocks (pooled)", "h": h, **R.moments(R.every_nth(rs, h).to_numpy())})
    rows.append({"series": "Nifty 50", "h": h, **R.moments(R.every_nth(R.label_returns(NC, NO, None, h, "close"), h).to_numpy())})
dist = pd.DataFrame(rows).set_index(["series", "h"])
display(dist.style.format({"n": "{:,.0f}", "mean": "{:.4%}", "std": "{:.3%}", "skew": "{:.2f}", "excess_kurt": "{:.1f}",
                           "share_beyond_3sd": "{:.2%}", "normal_beyond_3sd": "{:.2%}", "p01": "{:.2%}", "p99": "{:.2%}"}))

per_stock = pd.DataFrame({s: R.moments(r[s]) for s in r.columns}).T
print("Per-stock daily returns (median and range across 50 stocks):")
display(per_stock[["std", "skew", "excess_kurt", "share_beyond_3sd"]].astype(float).describe().loc[["min", "50%", "max"]])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
z = (r - r.mean()) / r.std()
zz = z.to_numpy().ravel(); zz = zz[~np.isnan(zz)]
grid = np.linspace(-8, 8, 400)
axes[0].hist(zz, bins=200, range=(-8, 8), density=True, alpha=0.7, label="stocks, standardised daily")
axes[0].plot(grid, stats.norm.pdf(grid), "crimson", label="normal")
axes[0].set_yscale("log"); axes[0].set_ylim(1e-5, 1); axes[0].legend(fontsize=8)
axes[0].set_title("Daily returns vs normal (log density)", fontsize=10)
q = np.linspace(0.0005, 0.9995, 400)
axes[1].plot(stats.norm.ppf(q), np.quantile(zz, q), ".", ms=3, label="stocks")
nz = ((rn - rn.mean()) / rn.std()).dropna()
axes[1].plot(stats.norm.ppf(q), np.quantile(nz, q), ".", ms=3, label="Nifty")
axes[1].plot([-4, 4], [-4, 4], "k--", lw=1); axes[1].legend(fontsize=8)
axes[1].set_title("QQ plot vs normal (standardised)", fontsize=10); axes[1].set_xlabel("normal quantile")
for h in HORIZONS:
    x = R.every_nth(ret_h[h], h).to_numpy().ravel(); x = x[~np.isnan(x)]
    axes[2].hist(x, bins=120, range=(-0.3, 0.3), density=True, histtype="step", label=f"{h} day")
axes[2].legend(fontsize=8); axes[2].set_title("Stock log returns by horizon", fontsize=10)
fig.tight_layout(); save(fig, "02_return_distributions"); plt.show()

## 2. Base rates: share of up moves

This is the class balance of the direction labels and the accuracy of always guessing the majority class.
Two label definitions are compared (the choice is made in T0):
- **close**: close(d−1) → close(d+h−1), the usual time-series convention;
- **open**: open(d) → close(d+h−1), which matches a trade entered at the 09:15 open of d.

Windows overlap here (every day is a forecast day), as they will in the labels.

In [ ]:
lab = {(k, h): R.label_returns(C, O, MASK, h, k) for k in ("close", "open") for h in HORIZONS}
nlab = {(k, h): R.label_returns(NC, NO, None, h, k)["NIFTY"] for k in ("close", "open") for h in HORIZONS}

def up_share(x):
    v = x.to_numpy().ravel() if isinstance(x, pd.DataFrame) else x.to_numpy()
    v = v[~np.isnan(v)]
    return (v > 0).mean()

rows = []
for (k, h), x in lab.items():
    rows.append({"label": k, "h": h,
                 "stocks train": up_share(x[is_train]), "stocks 2023": up_share(x[~is_train]),
                 "Nifty train": up_share(nlab[(k, h)][nlab[(k, h)].index <= TRAIN_END]),
                 "Nifty 2023": up_share(nlab[(k, h)][nlab[(k, h)].index > TRAIN_END]),
                 "exactly 0 (stocks)": float((x.to_numpy() == 0).sum() / np.isfinite(x.to_numpy()).sum())})
base = pd.DataFrame(rows).set_index(["label", "h"])
display(base.style.format("{:.1%}"))

by_year = pd.DataFrame({f"{k} {h}d": x.groupby(x.index.year).apply(up_share) for (k, h), x in lab.items()})
display(by_year.style.format("{:.1%}"))
by_stock = pd.DataFrame({f"{k} {h}d": x.loc[is_train, TRAIN_STOCKS].apply(lambda s: (s.dropna() > 0).mean()) for (k, h), x in lab.items()})
print(f"Per stock with >= 250 train days ({len(TRAIN_STOCKS)} stocks), train 2016-2022 (min / median / max):")
display(by_stock.describe().loc[["min", "50%", "max"]].style.format("{:.1%}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for col in by_year.columns:
    axes[0].plot(by_year.index, by_year[col], marker="o", ms=3, ls="-" if col.startswith("close") else "--", label=col)
axes[0].axhline(0.5, color="k", lw=0.8); axes[0].legend(fontsize=7, ncol=2)
axes[0].set_title("Share of up moves per year, 50 stocks pooled", fontsize=10)
axes[1].boxplot([by_stock[c] for c in by_stock.columns], tick_labels=by_stock.columns)
axes[1].axhline(0.5, color="k", lw=0.8); axes[1].tick_params(axis="x", labelrotation=45, labelsize=8)
axes[1].set_title("Share of up moves per stock, train 2016-2022", fontsize=10)
fig.tight_layout(); save(fig, "02_base_rates"); plt.show()

## 3. Autocorrelation of returns and absolute returns, lags 1–20

Return autocorrelation is the linear predictability ARIMA can use; absolute-return autocorrelation is volatility clustering. The dashed band is ±1.96/√n for one stock's series.

In [ ]:
LAGS = 20
print("stocks with >= 250 train returns:", len(TRAIN_STOCKS), "| left out:", sorted(set(r.columns) - set(TRAIN_STOCKS)))
train_r = r.loc[is_train, TRAIN_STOCKS]
acf_r = pd.DataFrame({s: acf(train_r[s].dropna(), nlags=LAGS, fft=True)[1:] for s in TRAIN_STOCKS}, index=range(1, LAGS + 1))
acf_a = pd.DataFrame({s: acf(train_r[s].dropna().abs(), nlags=LAGS, fft=True)[1:] for s in TRAIN_STOCKS}, index=range(1, LAGS + 1))
rn_tr = rn[rn.index <= TRAIN_END].dropna()
acf_n, acf_na = acf(rn_tr, nlags=LAGS, fft=True)[1:], acf(rn_tr.abs(), nlags=LAGS, fft=True)[1:]
band = 1.96 / np.sqrt(train_r.notna().sum().median())

lb = pd.DataFrame({s: acorr_ljungbox(train_r[s].dropna(), lags=[10]).iloc[0] for s in TRAIN_STOCKS}).T
lb_abs = pd.DataFrame({s: acorr_ljungbox(train_r[s].dropna().abs(), lags=[10]).iloc[0] for s in TRAIN_STOCKS}).T
summary = pd.DataFrame({
    "median lag-1 ACF": [acf_r.loc[1].median(), acf_a.loc[1].median()],
    "Nifty lag-1 ACF": [acf_n[0], acf_na[0]],
    "stocks with |lag-1| > band": [int((acf_r.loc[1].abs() > band).sum()), int((acf_a.loc[1].abs() > band).sum())],
    "median mean |ACF| lags 1-20": [acf_r.abs().mean().median(), acf_a.abs().mean().median()],
    "Ljung-Box(10) p<0.05, stocks": [int((lb.lb_pvalue < 0.05).sum()), int((lb_abs.lb_pvalue < 0.05).sum())],
}, index=["returns", "|returns|"])
display(summary)
rn_ex = rn_tr[rn_tr.index.year != 2020]
a_ex = acf(rn_ex, nlags=LAGS, fft=True)[1:]
print(f"Nifty ACF at lags 5/6: {acf_n[4]:+.3f}/{acf_n[5]:+.3f}; without 2020: {a_ex[4]:+.3f}/{a_ex[5]:+.3f}; "
      f"max |ACF| lags 1-20 without 2020: {np.abs(a_ex).max():.3f}")
print(f"band ±{band:.3f}; stocks with negative lag-1 return ACF: {int((acf_r.loc[1] < 0).sum())} of {len(TRAIN_STOCKS)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, a, an, title in [(axes[0], acf_r, acf_n, "returns"), (axes[1], acf_a, acf_na, "absolute returns")]:
    ax.fill_between(a.index, a.quantile(0.1, axis=1), a.quantile(0.9, axis=1), alpha=0.3, label="stocks 10-90%")
    ax.plot(a.index, a.median(axis=1), "o-", ms=3, label="stocks median")
    ax.plot(a.index, an, "s--", ms=3, color="crimson", label="Nifty 50")
    ax.axhline(band, color="k", ls=":", lw=1); ax.axhline(-band, color="k", ls=":", lw=1); ax.axhline(0, color="k", lw=0.6)
    ax.set_title(f"Autocorrelation of daily {title}, train 2016-2022", fontsize=10); ax.set_xlabel("lag (days)")
    ax.legend(fontsize=8)
fig.tight_layout(); save(fig, "02_autocorrelation"); plt.show()

## 4. Stationarity: ADF test on log prices vs log returns

The log price is rebuilt as the cumulative sum of the masked returns (ex-date jumps removed), so unadjusted splits don't fake a break. Train period only.

In [ ]:
import warnings

def adf_p(x):
    x = x.dropna()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # statsmodels warns about p-value table interpolation at the extremes
        return adfuller(x, autolag="AIC")[1] if len(x) > 100 else np.nan

logp = train_r.fillna(0).cumsum().where(C.loc[is_train, TRAIN_STOCKS].notna())
adf = pd.DataFrame({"log price p": logp.apply(adf_p), "log return p": train_r.apply(adf_p)})
adf.loc["NIFTY"] = [adf_p(np.log(NC["NIFTY"][NC.index <= TRAIN_END])), adf_p(rn_tr)]
res = pd.DataFrame({"series": [len(adf)] * 2,
                    "reject unit root at 5%": [int((adf["log price p"] < 0.05).sum()), int((adf["log return p"] < 0.05).sum())],
                    "median p-value": [adf["log price p"].median(), adf["log return p"].median()]},
                   index=["log price", "log return"])
display(res)
print("Log prices that reject a unit root:", adf.index[adf["log price p"] < 0.05].tolist())

## 5. Overnight gap vs intraday move

Trades enter at the open, and a stop can only fill at the open if the price gaps through it. The gap is
measured in units of the previous day's 14-day ATR (true range as a share of price), the usual stop unit.

In [ ]:
gap, intra = R.gap_intraday(O, C, MASK)
prev_c = C.shift(1)
tr = pd.concat([(H - L), (H - prev_c).abs(), (L - prev_c).abs()], keys=["a", "b", "c"]).groupby(level=1).max()
tr = (tr / prev_c).mask(MASK).reindex(C.index)
atr = tr.rolling(14, min_periods=14).mean().shift(1)            # known before the open of d
gap_atr = (gap.abs() / atr)[is_train]

g, i_ = gap[is_train], intra[is_train]
ok = g.notna() & i_.notna()
var_share = (g.where(ok).var() / (g.where(ok).var() + i_.where(ok).var()))
gv = gap_atr.to_numpy().ravel(); gv = gv[~np.isnan(gv)]
gaps = pd.Series({
    "median |gap| (% of price)": g.abs().stack().median(),
    "median |intraday| (% of price)": i_.abs().stack().median(),
    "gap share of variance, median stock": var_share.median(),
    "corr(gap, intraday), pooled": pd.concat([g.stack(), i_.stack()], axis=1).dropna().corr().iloc[0, 1],
    "median ATR(14) (% of price)": atr[is_train].stack().median(),
    "days |gap| > 0.5 ATR": (gv > 0.5).mean(), "days |gap| > 1 ATR": (gv > 1).mean(),
    "days |gap| > 2 ATR": (gv > 2).mean(),
})
display(gaps.to_frame("train 2016-2022").style.format("{:.2%}"))
ng, ni = R.gap_intraday(NO, NC)
print(f"Nifty: gap share of variance {ng.NIFTY.var() / (ng.NIFTY.var() + ni.NIFTY.var()):.1%}, "
      f"median |gap| {ng.NIFTY.abs().median():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(gv, bins=120, range=(0, 3), color="#4C72B0")
for x in (0.5, 1, 2):
    axes[0].axvline(x, color="crimson", ls="--", lw=1)
axes[0].set_yscale("log"); axes[0].set_xlabel("|open - previous close| / ATR(14)")
axes[0].set_title("Overnight gap in ATR units, stock-days 2016-2022", fontsize=10)
var_share.sort_values().plot.bar(ax=axes[1], color="#55A868", width=0.8)
axes[1].set_title("Share of daily return variance from the overnight gap", fontsize=10)
axes[1].tick_params(axis="x", labelsize=6)
fig.tight_layout(); save(fig, "02_gap_intraday"); plt.show()

## 6. Volatility and the historical-volatility baseline (task 16.5)

The baseline forecast for day d is the 20-day realised volatility up to d−1 (std of daily log returns, not
annualised). Targets: the std of daily log returns over d..d+4 and d..d+9 (the volatility label). Errors
are compared with a constant forecast (each stock's train mean) as a floor.

In [ ]:
rv20 = R.realised_vol(r, 20).shift(1)
rows = []
for h in (5, 10):
    fv = R.forward_vol(r, h)
    const = fv[is_train].mean()                         # per-stock train mean
    for period, sel in [("train", is_train), ("2023", ~is_train)]:
        p, a = rv20[sel].stack(), fv[sel].stack()
        d = pd.concat([p, a, const.reindex(a.index.get_level_values(1)).set_axis(a.index)], axis=1,
                      keys=["pred", "act", "const"]).dropna()
        rows.append({"h": h, "period": period, "n": len(d), "pearson": d.pred.corr(d.act),
                     "spearman": d.pred.corr(d.act, method="spearman"),
                     "MAE hist-vol": (d.pred - d.act).abs().mean(), "MAE constant": (d.const - d.act).abs().mean(),
                     "bias (pred - actual)": (d.pred - d.act).mean(), "mean actual": d.act.mean()})
hv = pd.DataFrame(rows).set_index(["h", "period"])
display(hv.style.format({"n": "{:,.0f}", "pearson": "{:.2f}", "spearman": "{:.2f}", "MAE hist-vol": "{:.3%}",
                         "MAE constant": "{:.3%}", "bias (pred - actual)": "{:+.3%}", "mean actual": "{:.3%}"}))

vol_year = pd.DataFrame({"stocks median rv20 (ann.)": R.realised_vol(r, 20).median(axis=1).groupby(r.index.year).mean() * np.sqrt(252),
                         "Nifty rv20 (ann.)": R.realised_vol(rn.to_frame(), 20)["NIFTY"].groupby(rn.index.year).mean() * np.sqrt(252)})
display(vol_year.style.format("{:.1%}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ann = np.sqrt(252)
axes[0].plot(R.realised_vol(r, 20).median(axis=1) * ann, lw=1, label="stocks, median 20-day vol")
axes[0].plot(R.realised_vol(rn.to_frame(), 20)["NIFTY"] * ann, lw=1, label="Nifty 50 20-day vol")
axes[0].axvline(pd.Timestamp("2023-01-01"), color="k", ls="--", lw=1)
axes[0].set_title("20-day realised volatility (annualised)", fontsize=10); axes[0].legend(fontsize=8)
fv10 = R.forward_vol(r, 10)
d = pd.concat([rv20[is_train].stack(), fv10[is_train].stack()], axis=1, keys=["p", "a"]).dropna()
axes[1].hexbin(np.log10(d.p), np.log10(d.a), gridsize=60, bins="log", cmap="viridis")
lim = [np.log10(0.003), np.log10(0.08)]
axes[1].plot(lim, lim, "w--", lw=1); axes[1].set_xlim(lim); axes[1].set_ylim(lim)
axes[1].set_xlabel("log10 past 20-day vol (forecast)"); axes[1].set_ylabel("log10 next 10-day vol")
axes[1].set_title("Historical-volatility forecast vs outcome, train", fontsize=10)
fig.tight_layout(); save(fig, "02_volatility"); plt.show()

## 7. Market link: beta, correlation and sectors

Daily returns, train period, against the Nifty 50.

In [ ]:
bc = R.beta_corr(r[is_train], rn[rn.index <= TRAIN_END])
bc["sector"] = sector.reindex(bc.index)
display(bc[["beta", "corr"]].astype(float).describe().loc[["min", "50%", "max"]])
display(bc.sort_values("beta").iloc[[0, 1, 2, -3, -2, -1]])

order = bc.sort_values(["sector", "beta"]).index
cm = r[is_train][order].corr(min_periods=250)
sec = bc.loc[order, "sector"]
same = np.equal.outer(sec.to_numpy(), sec.to_numpy())
offdiag = ~np.eye(len(order), dtype=bool)
within, across = cm.to_numpy()[same & offdiag], cm.to_numpy()[~same]
print(f"mean pairwise correlation: within sector {np.nanmean(within):.2f}, across sectors {np.nanmean(across):.2f}")
sec_tab = bc.groupby("sector").agg(stocks=("beta", "size"), beta=("beta", "mean"), corr=("corr", "mean")).sort_values("beta")
display(sec_tab)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.4, 1]})
cm.index = [f"{s} ({sec[s][:12]})" for s in cm.index]
heatmap(axes[0], cm, "Correlation of daily returns, train (ordered by sector)", cmap="RdBu_r", vmin=-1, vmax=1)
b = bc.loc[order[::-1]]
axes[1].barh(range(len(b)), b.beta.astype(float), color="#4C72B0")
axes[1].set_yticks(range(len(b)), b.index, fontsize=7); axes[1].axvline(1, color="k", lw=0.8)
axes[1].set_title("Beta to Nifty 50, train", fontsize=10)
fig.tight_layout(); save(fig, "02_market_link"); plt.show()

## 8. Regimes: Nifty trend and India VIX level

Labels come from `src/features/regime.py` with the `regime` settings in config.yaml. Each label for day d
uses closes up to d−1. The VIX cut-points are terciles of 2016–2022 only. The 200-day EMA needs 200 days of
history, so trend labels start in October 2016.

In [ ]:
reg = G.regime_table(NC["NIFTY"], vix, cfg["regime"])
print("VIX tercile cut-points (train 2016-2022):", [round(b, 2) for b in reg.attrs["vix_bands"]])
reg["combo"] = reg.trend.astype(str) + " / " + reg.vix.astype(str)

up5 = (lab[("close", 5)] > 0).where(lab[("close", 5)].notna())
frame = pd.DataFrame({"nifty_r": rn, "stock_abs_r": r.abs().mean(axis=1), "stock_up5": up5.mean(axis=1),
                      "trend": reg.trend, "vix": reg.vix}).loc[:TRAIN_END]

def by(col):
    g = frame.dropna(subset=[col]).groupby(col)
    return pd.DataFrame({"days": g.size(), "Nifty mean daily (bp)": g.nifty_r.mean() * 1e4,
                         "Nifty daily std": g.nifty_r.std(), "stocks mean |r|": g.stock_abs_r.mean(),
                         "stocks 5d up share": g.stock_up5.mean()})
regime_stats = pd.concat({"trend": by("trend"), "vix": by("vix")})
display(regime_stats.style.format({"days": "{:,.0f}", "Nifty mean daily (bp)": "{:+.1f}", "Nifty daily std": "{:.2%}",
                                   "stocks mean |r|": "{:.2%}", "stocks 5d up share": "{:.1%}"}))
frame["combo"] = reg.combo
display(frame[frame.trend.notna() & frame.vix.notna()].groupby("combo")
        .agg(days=("nifty_r", "size"), nifty_bp=("nifty_r", lambda x: x.mean() * 1e4), up5=("stock_up5", "mean")))

mix = pd.concat({"trend": pd.crosstab(reg.index.year, reg.trend, normalize="index"),
                 "vix": pd.crosstab(reg.index.year, reg.vix, normalize="index")[["low", "mid", "high"]]}, axis=1)
display(mix.style.format("{:.0%}"))

fig, axes = plt.subplots(2, 2, figsize=(15, 7), gridspec_kw={"width_ratios": [2.2, 1]})
ema200 = G.ema(NC["NIFTY"], cfg["regime"]["trend_ema_span"])
axes[0, 0].plot(NC["NIFTY"], lw=1, label="Nifty 50"); axes[0, 0].plot(ema200, lw=1, label="200-day EMA")
below = reg.trend == "below_ema"
axes[0, 0].fill_between(reg.index, 0, 1, where=below, transform=axes[0, 0].get_xaxis_transform(), color="crimson",
                        alpha=0.12, label="below EMA (lagged label)")
axes[0, 0].legend(fontsize=8); axes[0, 0].set_title("Trend regime", fontsize=10)
axes[1, 0].plot(vix, lw=1, color="#8172B2")
for b in reg.attrs["vix_bands"]:
    axes[1, 0].axhline(b, color="k", ls="--", lw=1)
axes[1, 0].set_title("India VIX with train tercile cut-points", fontsize=10)
for ax in axes[:, 0]:
    ax.axvline(pd.Timestamp("2023-01-01"), color="k", lw=0.8)
mix["trend"][["above_ema", "below_ema"]].plot.bar(stacked=True, ax=axes[0, 1], color=["#55A868", "crimson"], width=0.8)
axes[0, 1].set_title("Trend regime mix by year", fontsize=10); axes[0, 1].legend(fontsize=7)
mix["vix"].plot.bar(stacked=True, ax=axes[1, 1], color=["#55A868", "#CCB974", "crimson"], width=0.8)
axes[1, 1].set_title("VIX regime mix by year", fontsize=10); axes[1, 1].legend(fontsize=7)
for ax in axes[:, 1]:
    ax.set_xlabel("")
fig.tight_layout(); save(fig, "02_regimes"); plt.show()

## 9. Liquidity: turnover and zero-volume days

Turnover is traded value in ₹ (shown in ₹ crore = 10⁷).

In [ ]:
turn_cr = TURN / 1e7
liq = pd.DataFrame({"median daily turnover (Rs cr)": turn_cr.median(), "2023 median (Rs cr)": turn_cr[~is_train].median(),
                    "zero-volume days": (VOL == 0).sum(), "days": C.notna().sum()}).sort_values("median daily turnover (Rs cr)")
display(liq.iloc[[0, 1, 2, -3, -2, -1]])
print(f"median stock turnover: Rs {liq.iloc[:, 0].median():,.0f} cr/day; lowest: {liq.index[0]} Rs {liq.iloc[0, 0]:,.0f} cr; "
      f"zero-volume stock-days: {int(liq['zero-volume days'].sum())}")
yr = turn_cr.median(axis=1).groupby(turn_cr.index.year).median()
display(yr.to_frame("median stock turnover (Rs cr/day)").T.style.format("{:,.0f}"))

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})
liq.iloc[:, 0].plot.bar(ax=axes[0], color="#4C72B0", width=0.8, logy=True)
axes[0].set_title("Median daily turnover per stock, 2016-2023 (Rs crore, log scale)", fontsize=10)
axes[0].tick_params(axis="x", labelsize=7)
yr.plot.bar(ax=axes[1], color="#55A868"); axes[1].set_title("Median stock's daily turnover by year (Rs crore)", fontsize=10)
fig.tight_layout(); save(fig, "02_liquidity"); plt.show()

## 10. Train vs validation: does 2023 resemble 2016–2022?

In [ ]:
def period_stats(sel_r, sel_n, reg_sel):
    v = sel_r.to_numpy().ravel(); v = v[~np.isnan(v)]
    cmx = sel_r.corr(min_periods=100).to_numpy(); cmx = cmx[~np.eye(len(cmx), dtype=bool)]
    z = (v - v.mean()) / v.std()
    out = {"stock daily std": v.std(), "stock excess kurtosis": stats.kurtosis(v), "share beyond 3 sd": (np.abs(z) > 3).mean(),
           "Nifty daily std": sel_n.std(), "Nifty mean daily (bp)": sel_n.mean() * 1e4,
           "mean pairwise corr": np.nanmean(cmx)}
    for h in HORIZONS:
        x = lab[("close", h)].loc[sel_r.index]
        out[f"up share {h}d"] = up_share(x)
    for k in ("low", "mid", "high"):
        out[f"VIX {k} days"] = (reg_sel.vix == k).mean()
    out["below-EMA days"] = (reg_sel.trend == "below_ema").sum() / reg_sel.trend.notna().sum()
    return out

cmp_ = pd.DataFrame({
    "train 2016-2022": period_stats(r[is_train], rn[rn.index <= TRAIN_END], reg.loc[:TRAIN_END]),
    "train excl. 2020": period_stats(r[is_train & (r.index.year != 2020)], rn[(rn.index <= TRAIN_END) & (rn.index.year != 2020)],
                                     reg.loc[:TRAIN_END][reg.loc[:TRAIN_END].index.year != 2020]),
    "validation 2023": period_stats(r[~is_train], rn[rn.index > TRAIN_END], reg.loc["2023"]),
})
display(cmp_.style.format("{:.3f}"))
ks = stats.ks_2samp(r[is_train].stack().dropna(), r[~is_train].stack().dropna())
ks_std = stats.ks_2samp((r[is_train] / r[is_train].std()).stack().dropna(), (r[~is_train] / r[~is_train].std()).stack().dropna())
ex_adani = [c for c in r.columns if not c.startswith("ADANI")]
v23 = r.loc[~is_train, ex_adani].to_numpy().ravel()
print(f"2023 excess kurtosis without ADANIENT/ADANIPORTS: {stats.kurtosis(v23[~np.isnan(v23)]):.1f} "
      f"(with them: {cmp_.loc['stock excess kurtosis', 'validation 2023']:.1f})")
print(f"KS test, pooled daily returns train vs 2023: D={ks.statistic:.3f}; after scaling each stock by its own std: D={ks_std.statistic:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for lbl, sel in [("train 2016-2022", is_train), ("validation 2023", ~is_train)]:
    v = r[sel].to_numpy().ravel(); v = v[~np.isnan(v)]
    axes[0].hist(v, bins=200, range=(-0.1, 0.1), density=True, histtype="step", lw=1.2, label=lbl)
    rv = (R.realised_vol(r, 20)[sel] * np.sqrt(252)).to_numpy().ravel(); rv = rv[~np.isnan(rv)]
    axes[1].hist(rv, bins=120, range=(0, 1), density=True, histtype="step", lw=1.2, label=lbl)
axes[0].set_title("Daily stock returns", fontsize=10); axes[0].legend(fontsize=8)
axes[1].set_title("20-day realised volatility (annualised)", fontsize=10); axes[1].legend(fontsize=8)
fig.tight_layout(); save(fig, "02_train_vs_val"); plt.show()

## Findings

Raw closes with 604 ex-date returns dropped (0.67% of stock-days); train = 2016–2022, validation = 2023.

1. **Returns have fat tails at every horizon.** Pooled daily stock returns have a std of 2.01%, skew of −0.20 and excess kurtosis of 11.6. 1.55% of days lie beyond 3 sd, 5.7× the 0.27% of a normal distribution. The Nifty is more negatively skewed: std 1.07%, skew −1.47, kurtosis 22.5. Tails stay heavy at 5 and 10 days (kurtosis 7.0 and 5.4, about 1.5% beyond 3 sd).
2. **Base rates depend on the label definition.** Share of up moves, stocks pooled, train, at 1 / 5 / 10 days:
   - close(d−1)→close(d+h−1): 50.7% / 53.4% / 55.0%
   - open(d)→close(d+h−1): 45.8% / 51.5% / 53.6%

   The 1-day open label is down 54.2% of the time: stocks drift down during the session, and the positive drift happens overnight.
3. **Base rates move a lot between years and stocks.** The 10-day close label ranges from 48.8% up (2018) to 59.7% (2017), and from 40.2% to 62.4% across the 48 stocks with train history. In 2023 the shares rise to 53.2% / 59.0% / 62.4% (close) and 46.8% / 56.4% / 61.1% (open).
4. **There is almost no linear predictability in returns.** The median lag-1 autocorrelation is −0.012. Only 9 of 48 stocks fall outside ±0.047, although Ljung-Box(10) rejects for 28 of 48. The Nifty's large lag-5/6 values (+0.115/−0.121) come from 2020: without it they are +0.000/−0.028, and the largest is 0.058.
5. **Volatility clustering is strong.** The lag-1 autocorrelation of absolute returns is 0.175 for the median stock and 0.283 for the Nifty. All 48 stocks are significant, and it stays above 0.05 out to lag 20.
6. **Log prices have a unit root; log returns are stationary.** The ADF test rejects a unit root for 1 of 49 log-price series (TATACONSUM, a short 2020–22 series), with a median p of 0.71. It rejects for 49 of 49 return series (median p ≈ 1e-29).
7. **Overnight gaps are small but not negligible.** The median absolute gap is 0.39% of price, against 0.99% for the intraday move. Gaps carry 20% of the daily variance for the median stock (37% for the Nifty). With a median ATR(14) of 2.67%, a gap exceeds 0.5 / 1 / 2 ATR on 6.8% / 1.26% / 0.18% of stock-days. A 1-ATR stop is jumped at the open about once in 80 stock-days.
8. **The historical-volatility baseline (task 16.5) is informative but not easy to beat.** Past 20-day volatility correlates 0.43 / 0.48 with the next 5 / 10 days' volatility. In train its MAE (0.772% / 0.657%) is worse than a constant per-stock forecast (0.731% / 0.614%), because the 2020 shock makes trailing windows lag. In 2023 it beats the constant (0.601% vs 0.794% at 5 days, 0.514% vs 0.735% at 10). It over-forecasts by 0.14 / 0.07 points, partly because a 5-day std is biased low.
9. **Market link.** The median beta to the Nifty is 0.95, ranging from 0.47 (CIPLA) to 1.51 (TMPV). ETERNAL's 1.68 rests on only 358 days. The median correlation is 0.52. Pairwise correlation is 0.42 within a sector and 0.28 across sectors. By sector:
   - lowest betas: Healthcare 0.59, Power 0.64, FMCG 0.70, IT 0.75
   - highest betas: Metals 1.36, Financials 1.14
10. **Regimes separate volatility clearly.** The VIX tercile cut-points (train) are 14.79 and 18.35. The Nifty's daily std is 0.85% above its 200-day EMA vs 1.97% below. By VIX regime it is 0.63% / 0.90% / 1.60% (low / mid / high). Stocks' 5-day up share is higher after stress: 56.5% below the EMA vs 52.5% above, and 55.9% at high VIX vs 50.8% at low. Train has 1,247 above-EMA days and 287 below. Trend labels start in Oct 2016 because the EMA needs 200 days.
11. **All 50 stocks are liquid.** Median turnover is ₹194 crore a day. The lowest are TRENT (₹16 crore), BSE (₹17 crore) and MAXHEALTH (₹45 crore). There are 0 zero-volume stock-days. The median stock's turnover rose from ₹99 crore (2016) to ₹259 crore (2023).
12. **2023 is calmer and more one-directional than 2016–2022:**

    | Measure | Train 2016–2022 | 2023 |
    |---|---|---|
    | Stock daily std | 2.1% | 1.6% |
    | Nifty daily std | 1.1% | 0.6% |
    | Days at low VIX | 33% | 89% |
    | Days at high VIX | 34% | 0% |
    | Days below the 200-day EMA | 19% | 9% |
    | Mean pairwise correlation | 0.29 | 0.17 |

    2023's excess kurtosis of 41.9 is almost all the Adani fall (5.0 without ADANIENT and ADANIPORTS). KS distance between the periods: 0.075 on raw returns, 0.038 after scaling each stock by its own std.

### What this means for labels, features and baselines

- **Labels.** Use the open(d)→close(d+h−1) label, which matches the trade entry, and record the class balance per horizon. The 1-day label is imbalanced (45.8% up), and 5- and 10-day labels lean up, especially in 2023. Report balanced accuracy, F1 and Brier score, not raw accuracy alone.
- **The accuracy bar to beat** is the constant majority guess for the period being scored. On the open label that is 54.2% / 51.5% / 53.6% in train (1 / 5 / 10 days) and 53.2% / 56.4% / 61.1% in 2023. A model that always says "up" scores 61% at 10 days in 2023.
- **Features.** Volatility is predictable and returns are barely so. Volatility features (realised vol, ATR, India VIX, regime) should carry the most signal. Beta and sector explain shared moves. The higher up share after stress hints at mean reversion and is worth a feature (distance from EMA, recent drawdown).
- **ARIMA (16.6)** should be expected to forecast close to zero: there is little linear structure outside 2020. Its value is as a weak baseline.
- **Volatility baseline (16.5).** Compare models against both historical volatility and a constant per-stock volatility. Neither dominates across periods.
- **Backtest.** Fill stops at the open when the price gaps through them. That affects about 1.3% of stock-days for a 1-ATR stop.
- **Validation.** 2023 is a single low-volatility bull year with no high-VIX days, so it can't show how a model behaves in stress. Report validation results by regime, and add walk-forward folds inside 2016–2022 when tuning.